In [4]:
from pathlib import Path
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_curve
from difflib import SequenceMatcher
import ollama

data_path = Path("../data/smartdoc_demo")
new_documents_path = Path("../data/new_documents")

In [6]:
documents = []

for category_path in data_path.iterdir():
    if not category_path.is_dir():
        continue

    category = category_path.name

    for file_path in category_path.glob("*.txt"):
        text = file_path.read_text(encoding="utf-8", errors="ignore")
        documents.append({
            "filename": file_path.name,
            "text": text,
            "category": category,
            "path": str(file_path)
        })

df = pd.DataFrame(documents)

print("Nombre total de documents :", len(df))
print(df["category"].value_counts())

Nombre total de documents : 42
category
algorithmes_structures       4
biodiversite_ecosystemes     4
agriculture_urbanisme        3
bonnes_pratiques_dev         3
investissement               3
banque_credit                2
budget_epargne               2
climat                       2
destinations                 2
developpement_web            2
fiscalite_retraite           2
planification_voyage         2
voyages                      2
activites_voyage             1
assurance                    1
astronomie                   1
bases_de_donnees             1
energie                      1
gastronomie_voyage           1
hebergement_voyage           1
intelligence_artificielle    1
transport_voyage             1
Name: count, dtype: int64


In [7]:
embedding_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")

document_embeddings = embedding_model.encode(
    df["text"].tolist(),
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Dimension des embeddings :", document_embeddings.shape[1])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Dimension des embeddings : 768


In [8]:
category_document_embeddings = {}

for category in df["category"].unique():
    mask = df["category"].values == category
    category_document_embeddings[category] = list(document_embeddings[mask])

print("Catégories chargées :", list(category_document_embeddings.keys()))

Catégories chargées : ['activites_voyage', 'agriculture_urbanisme', 'algorithmes_structures', 'assurance', 'astronomie', 'banque_credit', 'bases_de_donnees', 'biodiversite_ecosystemes', 'bonnes_pratiques_dev', 'budget_epargne', 'climat', 'destinations', 'developpement_web', 'energie', 'fiscalite_retraite', 'gastronomie_voyage', 'hebergement_voyage', 'intelligence_artificielle', 'investissement', 'planification_voyage', 'transport_voyage', 'voyages']


In [9]:
def find_best_category(document_embedding, top_k=3):

    best_category = None
    best_score = -1
    best_document = None

    for category, category_embeddings_list in category_document_embeddings.items():

        category_array = np.atleast_2d(np.array(category_embeddings_list))

        scores = cosine_similarity(
            document_embedding.reshape(1, -1),
            category_array
        )[0]

        k = min(top_k, len(scores))
        top_scores = np.sort(scores)[-k:]
        max_score = float(np.mean(top_scores))

        max_index = int(np.argmax(scores))

        if max_score > best_score:
            best_score = max_score
            best_category = category

            category_df = df[df["category"] == category].reset_index(drop=True)
            if max_index < len(category_df):
                best_document = category_df.iloc[max_index]["filename"]
            else:
                best_document = "document ajouté dynamiquement"

    return best_category, best_score, best_document


def create_category(category_name, document_embedding):

    category_document_embeddings[category_name] = [document_embedding]

    category_path = data_path / category_name
    category_path.mkdir(parents=True, exist_ok=True)

    print(f"🆕 Nouvelle catégorie créée : {category_name}")
    print(f"📁 Dossier : {category_path}")


def generate_category_name(text, max_chars=800):

    excerpt = text[:max_chars]

    prompt = f"""Voici un extrait d'un document :

---
{excerpt}
---

Donne un nom de catégorie court (2 à 3 mots maximum) qui résume 
le sujet principal de ce document, en français.

Règles strictes :
- Réponds UNIQUEMENT avec le nom de la catégorie, rien d'autre
- Pas de phrase, pas d'explication, pas de ponctuation, pas de guillemets
- Utilise une forme générale (ex: "Recettes" et non "Recette de pâtes")

Nom de catégorie :"""

    response = ollama.chat(
        model="qwen2.5:3b",
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0.2}
    )

    category_name = response["message"]["content"].strip()
    category_name = category_name.split("\n")[0]
    category_name = category_name.strip('"\'.,: ')
    category_name = category_name.replace("/", "-").replace("\\", "-")

    if len(category_name.split()) > 4:
        category_name = " ".join(category_name.split()[:2])

    return category_name.capitalize()


def find_similar_existing_category_name(new_name, existing_categories, threshold=0.85):

    for existing in existing_categories:
        similarity = SequenceMatcher(None, new_name.lower(), existing.lower()).ratio()
        if similarity >= threshold:
            return existing

    return None

In [10]:
def smartdoc_categorize(file_path, threshold=0.353):  # ton seuil calibré actuel

    file_path = Path(file_path)

    if not file_path.exists():
        print("❌ Fichier introuvable :", file_path)
        return None

    text = file_path.read_text(encoding="utf-8", errors="ignore")

    if not text.strip():
        print("❌ Le document est vide :", file_path.name)
        return None

    embedding = embedding_model.encode([text], normalize_embeddings=True)[0]

    category, score, closest_document = find_best_category(embedding)

    print("\n" + "=" * 55)
    print("Document :", file_path.name)
    print("Document le plus proche :", closest_document)
    print("Meilleure catégorie :", category)
    print("Score :", round(score, 3))

    if score >= threshold:
        destination_category = category
        category_document_embeddings[category].append(embedding)
        print("→ Catégorie existante :", category)
        action = "existing_category"

    else:
        proposed_name = generate_category_name(text)
        print("🤖 Nom proposé par le LLM :", proposed_name)

        existing_match = find_similar_existing_category_name(
            proposed_name,
            category_document_embeddings.keys()
        )

        if existing_match:
            destination_category = existing_match
            category_document_embeddings[existing_match].append(embedding)
            print("→ Fusion avec catégorie existante :", existing_match)
            action = "existing_category"
        else:
            destination_category = proposed_name
            create_category(destination_category, embedding)
            print("→ Nouvelle catégorie créée :", destination_category)
            action = "new_category"

    destination_folder = data_path / destination_category
    destination_folder.mkdir(parents=True, exist_ok=True)
    destination_file = destination_folder / file_path.name
    file_path.rename(destination_file)

    print("📁 Fichier déplacé vers :", destination_file)

    return {
        "category": destination_category,
        "score": float(score),
        "action": action,
        "path": str(destination_file)
    }

In [11]:
similarity_matrix = cosine_similarity(document_embeddings)

same_category_scores = []
different_category_scores = []
n = len(df)

for i in range(n):
    for j in range(i + 1, n):
        score = similarity_matrix[i, j]
        if df.iloc[i]["category"] == df.iloc[j]["category"]:
            same_category_scores.append(score)
        else:
            different_category_scores.append(score)

same_category_scores = np.array(same_category_scores)
different_category_scores = np.array(different_category_scores)

labels = np.concatenate([
    np.ones(len(same_category_scores)),
    np.zeros(len(different_category_scores))
])
scores = np.concatenate([same_category_scores, different_category_scores])

fpr, tpr, thresholds = roc_curve(labels, scores)
youden_index = tpr - fpr
best_threshold = thresholds[np.argmax(youden_index)]

OPTIMAL_THRESHOLD = round(best_threshold, 3)
print("OPTIMAL_THRESHOLD :", OPTIMAL_THRESHOLD)

same_correct = (same_category_scores >= OPTIMAL_THRESHOLD).sum()
same_total = len(same_category_scores)
diff_correct = (different_category_scores < OPTIMAL_THRESHOLD).sum()
diff_total = len(different_category_scores)

print(f"Paires même catégorie bien détectées : {same_correct}/{same_total} ({100*same_correct/same_total:.1f}%)")
print(f"Paires catégorie différente bien détectées : {diff_correct}/{diff_total} ({100*diff_correct/diff_total:.1f}%)")

OPTIMAL_THRESHOLD : 0.442
Paires même catégorie bien détectées : 20/29 (69.0%)
Paires catégorie différente bien détectées : 734/832 (88.2%)


In [12]:
result_astronomie = smartdoc_categorize(
    "../data/new_documents/astronomie.txt"
)
print(result_astronomie)

❌ Fichier introuvable : ..\data\new_documents\astronomie.txt
None


In [10]:
result_python = smartdoc_categorize(
    "../data/new_documents/apprentissage_python.txt"
)
print(result_python)

❌ Fichier introuvable : ..\data\new_documents\apprentissage_python.txt
None


In [13]:
def chunk_text(text, chunk_size=300, overlap=50):
    """
    Découpe un texte en chunks (en mots) avec un léger 
    chevauchement pour ne pas couper une idée en plein milieu.
    """

    words = text.split()
    chunks = []

    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap  # overlap = chevauchement

    return chunks


# Construire la liste de tous les chunks, avec leur document d'origine
CHUNK_SIZE = 100
OVERLAP = 20

chunk_records = []

for _, row in df.iterrows():

    chunks = chunk_text(row["text"], chunk_size=CHUNK_SIZE, overlap=OVERLAP)

    for i, chunk in enumerate(chunks):
        chunk_records.append({
            "chunk_id": f"{row['filename']}_{i}",
            "filename": row["filename"],
            "category": row["category"],
            "chunk_index": i,
            "text": chunk
        })

chunks_df = pd.DataFrame(chunk_records)

print("Nombre total de chunks :", len(chunks_df))
print("\nRépartition par document :")
print(chunks_df.groupby("filename").size().sort_values(ascending=False).head(10))

Nombre total de chunks : 119

Répartition par document :
filename
agriculture_durable.txt      3
algorithmes_tri.txt          3
api_rest.txt                 3
assurance_habitation.txt     3
astronomie.txt               3
banque.txt                   3
budget_voyage.txt            3
biodiversite.txt             3
credit_immobilier.txt        3
changement_climatique.txt    3
dtype: int64


In [14]:
chunk_embeddings = embedding_model.encode(
    chunks_df["text"].tolist(),
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Shape des embeddings de chunks :", chunk_embeddings.shape)
print("Nombre de chunks indexés :", len(chunks_df))

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Shape des embeddings de chunks : (119, 768)
Nombre de chunks indexés : 119


In [15]:
def search_chunks(query, top_k=3):
    """
    Cherche les top_k chunks les plus pertinents pour une requête.
    """

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    similarities = cosine_similarity(
        query_embedding,
        chunk_embeddings
    )[0]

    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            "filename": chunks_df.iloc[idx]["filename"],
            "category": chunks_df.iloc[idx]["category"],
            "chunk_text": chunks_df.iloc[idx]["text"],
            "score": float(similarities[idx])
        })

    return results

In [16]:
results = search_chunks("Comment fonctionne l'énergie solaire ?")

for r in results:
    print(f"\n📄 {r['filename']} ({r['category']}) — score: {r['score']:.3f}")
    print(r["chunk_text"][:200], "...")


📄 energie_solaire.txt (energie) — score: 0.697
Une installation photovoltaïque transforme la lumière du soleil en électricité. Les cellules, généralement fabriquées à partir de silicium, produisent un courant continu lorsqu'elles reçoivent des pho ...

📄 energie_solaire.txt (energie) — score: 0.458
fonctionnement et les panneaux peuvent produire de l'électricité pendant plusieurs décennies. Leur installation nécessite toutefois des matériaux, un investissement initial et une surface correctement ...

📄 urbanisme_durable.txt (agriculture_urbanisme) — score: 0.388
concret dans la ville. Des arbres adaptés créent de l'ombre, ralentissent les eaux de pluie et réduisent localement la chaleur. Des sols perméables, des noues et des jardins de pluie facilitent l'infi ...


In [20]:
def rag_answer(query, top_k=3):
    """
    Pipeline RAG complet : recherche + génération de réponse.
    """

    # 1. Récupérer les chunks pertinents
    relevant_chunks = search_chunks(query, top_k=top_k)

    # 2. Construire le contexte à donner au LLM
    context = "\n\n".join([
        f"[Source: {c['filename']}]\n{c['chunk_text']}"
        for c in relevant_chunks
    ])

    # 3. Construire le prompt
    prompt = f"""Tu es l'assistant de SmartDoc, une application qui aide à retrouver 
des informations dans des documents personnels.

Voici des extraits de documents potentiellement pertinents :

{context}

Question de l'utilisateur : {query}

Consignes :
- Réponds UNIQUEMENT à partir des informations présentes dans les extraits ci-dessus
- Si les extraits ne permettent pas de répondre, dis-le clairement
- Cite le nom du fichier source pour chaque information utilisée
- Réponds en français, de façon claire et concise

Réponse :"""

    # 4. Appeler Qwen
    response = ollama.chat(
        model="qwen2.5:3b",
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0.3}
    )

    answer = response["message"]["content"].strip()

    return {
        "answer": answer,
        "sources": [c["filename"] for c in relevant_chunks],
        "chunks_used": relevant_chunks
    }

In [21]:
result = rag_answer("Quel budget prévoir pour un voyage, et comment le gérer financièrement ?")

print("💬 Réponse :\n")
print(result["answer"])
print("\n📚 Sources utilisées :", list(dict.fromkeys(result["sources"])))

💬 Réponse :

Pour un voyage, il est important de prévoir un budget qui tienne compte de tous les frais potentiels. Selon le document "budget_voyage.txt" de SmartDoc, il est recommandé de distinguer entre dépenses déjà payées et celles qui seront réglées sur place. Pour chaque journée de voyage, il est conseillé de prévoir une enveloppe raisonnable pour couvrir les dépenses.

Pour gérer le budget, il est recommandé de disposer de deux moyens de paiement séparés et d'avoir un peu d'argent liquide pour les situations où les commerces n'acceptent pas la carte. À la fin du séjour, il est utile de noter les dépenses réelles afin de préparer le prochain voyage sur une base concrète, plutôt que sur une estimation optimiste.

Ces conseils sont tirés du document "budget_voyage.txt" de SmartDoc.

📚 Sources utilisées : ['budget_voyage.txt', 'budget.txt']


In [18]:
result = rag_answer("Comment fonctionne l'énergie solaire ?")

print("💬 Réponse :\n")
print(result["answer"])
print("\n📚 Sources utilisées :", result["sources"])

💬 Réponse :

Les cellules produisent du courant continu (energie_solaire.txt). Cette lumière du soleil est transformée en électricité par des cellules, généralement fabriquées à partir de silicium, lorsqu'elles reçoivent des photons. Un onduleur convertit ensuite ce courant continu en courant alternatif utilisable par les appareils du logement ou injecté sur le réseau (energie_solaire.txt).

La production d'énergie solaire varie avec la saison, l'orientation du toit, l'inclinaison, l'ombrage et la météo (energie_solaire.txt). Elle est maximale autour du milieu de la journée. Cette source d'énergie n'émet pas de gaz à effet de serre pendant son fonctionnement (energie_solaire.txt).

L'installation d'une installation photovoltaïque nécessite des matériaux, un investissement initial et une surface correctement exposée (energie_solaire.txt). La production ne coïncide pas toujours avec la consommation, ce qui rend utile un pilotage des appareils, une batterie ou un contrat de revente selon 

In [19]:
result = rag_answer("Quel est le meilleur restaurant à Tokyo ?")

print("💬 Réponse :\n")
print(result["answer"])
print("\n📚 Sources utilisées :", list(dict.fromkeys(result["sources"])))

💬 Réponse :

Je suis désolé, mais il n'y a pas d'information sur le meilleur restaurant à Tokyo dans les extraits des documents fournis. (gastronomie_voyage.txt, paris.txt, voyage_en_train.txt)

📚 Sources utilisées : ['gastronomie_voyage.txt', 'paris.txt', 'voyage_en_train.txt']


In [22]:
# ============================================================
# JEU DE TEST POUR EVALUER LE RAG
# ============================================================

test_questions = [
    {
        "question": "Comment fonctionne l'énergie solaire ?",
        "expected_file": "energie_solaire.txt"
    },
    {
        "question": "Quelles sont les bonnes pratiques pour sécuriser son code ?",
        "expected_file": "securite_code.txt"
    },
    {
        "question": "Comment se déplacer en train pendant un voyage ?",
        "expected_file": "voyage_en_train.txt"
    },
    {
        "question": "Qu'est-ce que la biodiversité et pourquoi est-elle importante ?",
        "expected_file": "biodiversite.txt"
    },
        {
        "question": "Sur quels critères dois je me baser pour choisir un hotel ?",
        "expected_file": "hotel.txt"
    },
    # Ajoute-en 3-5 de plus, en piochant dans tes différentes catégories
]

In [23]:
def evaluate_retrieval(test_questions, top_k=3):

    correct = 0
    results_log = []

    for test in test_questions:

        retrieved = search_chunks(test["question"], top_k=top_k)
        retrieved_files = [r["filename"] for r in retrieved]

        found = test["expected_file"] in retrieved_files
        correct += found

        results_log.append({
            "question": test["question"],
            "expected": test["expected_file"],
            "retrieved": retrieved_files,
            "found": found
        })

    accuracy = correct / len(test_questions)

    print(f"Précision de la recherche (top_{top_k}) : {correct}/{len(test_questions)} ({accuracy*100:.1f}%)\n")

    for r in results_log:
        status = "✅" if r["found"] else "❌"
        print(f"{status} \"{r['question']}\"")
        print(f"   Attendu: {r['expected']} | Récupéré: {r['retrieved']}\n")

    return results_log

evaluation_results = evaluate_retrieval(test_questions)

Précision de la recherche (top_3) : 5/5 (100.0%)

✅ "Comment fonctionne l'énergie solaire ?"
   Attendu: energie_solaire.txt | Récupéré: ['energie_solaire.txt', 'energie_solaire.txt', 'urbanisme_durable.txt']

✅ "Quelles sont les bonnes pratiques pour sécuriser son code ?"
   Attendu: securite_code.txt | Récupéré: ['securite_code.txt', 'securite_code.txt', 'developpement_web.txt']

✅ "Comment se déplacer en train pendant un voyage ?"
   Attendu: voyage_en_train.txt | Récupéré: ['voyage_en_train.txt', 'paris.txt', 'tourisme_responsable.txt']

✅ "Qu'est-ce que la biodiversité et pourquoi est-elle importante ?"
   Attendu: biodiversite.txt | Récupéré: ['biodiversite.txt', 'forets.txt', 'biodiversite.txt']

✅ "Sur quels critères dois je me baser pour choisir un hotel ?"
   Attendu: hotel.txt | Récupéré: ['hotel.txt', 'hotel.txt', 'tourisme_responsable.txt']

